# BÀI THỰC HÀNH 4: MẠNG NEURAL HỒI QUY

<b>Hướng dẫn nộp bài:</b> Các bạn commit và push code lên github, sử dụng file txt đặt tên theo cú pháp <MSSV>.txt chứa đường link dẫn đến github của bài thực hành và nộp file txt này tên courses.

Bộ dữ liệu sử dụng: [PhoMT](https://drive.google.com/drive/folders/186OAOuSEYEDVcry7WP5UBdqECXo26QAb?usp=drive_link).

In [1]:
import json
import random
from collections import Counter
from pathlib import Path

import torch
from torch import nn
from torch.nn.utils.rnn import pad_sequence, pack_padded_sequence, pad_packed_sequence
from torch.utils.data import DataLoader

random.seed(42)
torch.manual_seed(42)
torch.set_num_threads(4)
device = torch.device('cpu')
print('Thiết bị:', device)

Thiết bị: cpu


In [2]:
with open('data/small-train.json', encoding='utf-8') as f:
    train_data = json.load(f)
with open('data/small-dev.json', encoding='utf-8') as f:
    dev_data = json.load(f)
with open('data/small-test.json', encoding='utf-8') as f:
    test_data = json.load(f)

print('Train:', len(train_data), 'Dev:', len(dev_data), 'Test:', len(test_data))

def tokenize(text):
    return text.lower().strip().split()

def make_vocab(data, language):
    counts = Counter(token for row in data for token in tokenize(row[language]))
    words = ['<pad>', '<unk>', '<bos>', '<eos>']
    words += [word for word, count in counts.most_common() if count >= 2 and word not in words][:14996]
    return {word: i for i, word in enumerate(words)}

src_vocab = make_vocab(train_data, 'english')
tgt_vocab = make_vocab(train_data, 'vietnamese')
tgt_words = list(tgt_vocab)
print('Từ vựng:', len(src_vocab), len(tgt_vocab))

def encode(text, vocab):
    return [vocab['<bos>']] + [vocab.get(word, 1) for word in tokenize(text)[:48]] + [vocab['<eos>']]

def make_batch(rows):
    src = [torch.tensor(encode(row['english'], src_vocab)) for row in rows]
    tgt = [torch.tensor(encode(row['vietnamese'], tgt_vocab)) for row in rows]
    return pad_sequence(src, batch_first=True), pad_sequence(tgt, batch_first=True)

train_loader = DataLoader(train_data, batch_size=64, shuffle=True, collate_fn=make_batch)
dev_loader = DataLoader(dev_data, batch_size=64, collate_fn=make_batch)
test_loader = DataLoader(test_data, batch_size=64, collate_fn=make_batch)

Train: 20000 Dev: 2000 Test: 2000


Từ vựng: 10084 4312


In [3]:
class Encoder(nn.Module):
    def __init__(self, vocab_size):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, 256, padding_idx=0)
        self.lstm = nn.LSTM(256, 256, num_layers=3, batch_first=True)

    def forward(self, src):
        lengths = src.ne(0).sum(dim=1).cpu()
        embedded = self.embedding(src)
        packed = pack_padded_sequence(embedded, lengths, batch_first=True, enforce_sorted=False)
        output, state = self.lstm(packed)
        output, _ = pad_packed_sequence(output, batch_first=True, total_length=src.size(1))
        return output, state

class Decoder(nn.Module):
    def __init__(self, vocab_size, attention):
        super().__init__()
        self.attention = attention
        self.embedding = nn.Embedding(vocab_size, 256, padding_idx=0)
        self.lstm = nn.LSTM(512 if attention == 'bahdanau' else 256, 256, num_layers=3, batch_first=True)
        if attention == 'bahdanau':
            self.attn_encoder = nn.Linear(256, 256, bias=False)
            self.attn_decoder = nn.Linear(256, 256, bias=False)
            self.attn_score = nn.Linear(256, 1, bias=False)
        if attention == 'luong':
            self.attn_encoder = nn.Linear(256, 256, bias=False)
        if attention != 'none':
            self.combine = nn.Linear(512, 256)
        self.output = nn.Linear(256, vocab_size)

    def forward(self, token, state, encoder_output, mask):
        embedded = self.embedding(token).unsqueeze(1)
        if self.attention == 'bahdanau':
            scores = self.attn_score(torch.tanh(self.attn_encoder(encoder_output) + self.attn_decoder(state[0][-1]).unsqueeze(1))).squeeze(-1)
            weights = torch.softmax(scores.masked_fill(~mask, -1e9), dim=1)
            context = torch.bmm(weights.unsqueeze(1), encoder_output)
            embedded = torch.cat((embedded, context), dim=2)
        output, state = self.lstm(embedded, state)
        if self.attention == 'luong':
            scores = torch.bmm(self.attn_encoder(encoder_output), output.transpose(1, 2)).squeeze(-1)
            weights = torch.softmax(scores.masked_fill(~mask, -1e9), dim=1)
            context = torch.bmm(weights.unsqueeze(1), encoder_output)
        if self.attention != 'none':
            output = torch.tanh(self.combine(torch.cat((output, context), dim=2)))
        return self.output(output.squeeze(1)), state

class Translator(nn.Module):
    def __init__(self, attention):
        super().__init__()
        self.encoder = Encoder(len(src_vocab))
        self.decoder = Decoder(len(tgt_vocab), attention)

    def forward(self, src, tgt):
        encoder_output, state = self.encoder(src)
        mask = src.ne(0)
        outputs = []
        for step in range(tgt.size(1) - 1):
            output, state = self.decoder(tgt[:, step], state, encoder_output, mask)
            outputs.append(output)
        return torch.stack(outputs, dim=1)

    def translate(self, src, max_length=50):
        encoder_output, state = self.encoder(src)
        mask = src.ne(0)
        token = torch.full((src.size(0),), 2, dtype=torch.long, device=src.device)
        finished = torch.zeros(src.size(0), dtype=torch.bool, device=src.device)
        result = []
        for _ in range(max_length):
            output, state = self.decoder(token, state, encoder_output, mask)
            token = output.argmax(dim=1)
            token = token.masked_fill(finished, 0)
            result.append(token)
            finished |= token.eq(3)
            if finished.all():
                break
        return torch.stack(result, dim=1)

In [4]:
loss_fn = nn.CrossEntropyLoss(ignore_index=0)

def epoch(model, loader, optimizer=None):
    model.train(optimizer is not None)
    total_loss = 0
    total_tokens = 0
    with torch.set_grad_enabled(optimizer is not None):
        for src, tgt in loader:
            src, tgt = src.to(device), tgt.to(device)
            if optimizer is not None:
                optimizer.zero_grad()
            output = model(src, tgt)
            target = tgt[:, 1:]
            loss = loss_fn(output.reshape(-1, output.size(-1)), target.reshape(-1))
            if optimizer is not None:
                loss.backward()
                nn.utils.clip_grad_norm_(model.parameters(), 1)
                optimizer.step()
            tokens = target.ne(0).sum().item()
            total_loss += loss.item() * tokens
            total_tokens += tokens
    return total_loss / total_tokens

def rouge_l(prediction, reference):
    previous = [0] * (len(reference) + 1)
    for word in prediction:
        current = [0]
        for j, ref in enumerate(reference, 1):
            current.append(previous[j - 1] + 1 if word == ref else max(previous[j], current[-1]))
        previous = current
    overlap = previous[-1]
    if not prediction or not reference or overlap == 0:
        return 0.0
    precision = overlap / len(prediction)
    recall = overlap / len(reference)
    return 2 * precision * recall / (precision + recall)

def evaluate(model, loader):
    model.eval()
    scores = []
    with torch.no_grad():
        for src, tgt in loader:
            predictions = model.translate(src.to(device)).cpu().tolist()
            for prediction, reference in zip(predictions, tgt.tolist()):
                pred_words = []
                for index in prediction:
                    if index == 3:
                        break
                    if index not in (0, 2):
                        pred_words.append(tgt_words[index])
                ref_words = tokenize(test_data[len(scores)]['vietnamese'])
                scores.append(rouge_l(pred_words, ref_words))
    return sum(scores) / len(scores)

def train_and_evaluate(attention):
    model = Translator(attention).to(device)
    optimizer = torch.optim.Adam(model.parameters(), lr=0.001)
    for i in range(1):
        train_loss = epoch(model, train_loader, optimizer)
        dev_loss = epoch(model, dev_loader)
        print('Epoch:', i + 1, 'Train loss:', round(train_loss, 4), 'Dev loss:', round(dev_loss, 4), flush=True)
    score = evaluate(model, test_loader)
    print('ROUGE-L:', round(score, 4), flush=True)
    return model, score

#### Bài 1: Xây dựng kiến trúc Encoder-Decoder gồm 3 lớp LSTM cho module encoder và 3 lớp LSTM cho module decoder, với hidden size là 256, cho bài toán dịch máy từ tiếng Anh sang tiếng Việt. Huấn luyện mô hình này trên bộ dữ liệu PhoMT sử dụng Adam làm phương thức tối ưu tham số. Đánh giá độ hiệu quả của mô hình sử dụng độ đo ROUGE-L.

In [5]:
model_none, score_none = train_and_evaluate('none')

Epoch: 1 Train loss: 6.1082 Dev loss: 6.0476


ROUGE-L: 0.1035


#### Bài 2: Xây dựng kiến trúc Encoder-Decoder gồm 3 lớp LSTM cho module encoder và 3 lớp LSTM cho module decoder, với hidden size là 256, cho bài toán dịch máy từ tiếng Anh sang tiếng Việt. Module decoder được trang bị kỹ thuật attention theo mô tả của nghiên cứu "[Neural Machine Translation by Jointly Learning to Align and Translate](https://arxiv.org/abs/1409.0473)". Huấn luyện mô hình này trên bộ dữ liệu PhoMT sử dụng Adam làm phương thức tối ưu tham số. Đánh giá độ hiệu quả của mô hình sử dụn độ đo ROUGE-L.

In [6]:
model_bahdanau, score_bahdanau = train_and_evaluate('bahdanau')

Epoch: 1 Train loss: 6.125 Dev loss: 6.1829


ROUGE-L: 0.1019


#### Bài 3: Xây dựng kiến trúc Encoder-Decoder gồm 3 lớp LSTM cho module encoder và 3 lớp LSTM cho module decoder, với hidden size là 256, cho bài toán dịch máy từ tiếng Anh sang tiếng Việt. Module decoder được trang bị kỹ thuật attention theo mô tả của nghiên cứu "[Effective Approaches to Attention-based Neural Machine Translation](https://arxiv.org/abs/1508.04025)". Huấn luyện mô hình này trên bộ dữ liệu PhoMT sử dụng Adam làm phương thức tối ưu tham số. Đánh giá độ hiệu quả của mô hình sử dụn độ đo ROUGE-L.

In [7]:
model_luong, score_luong = train_and_evaluate('luong')

Epoch: 1 Train loss: 6.1034 Dev loss: 6.0778


ROUGE-L: 0.1099
